In [1]:
import torch
import random
import math
from torch import empty

In [45]:
# just to be sure
torch.set_grad_enabled(False);

In [3]:
# Mother class of all modules
class Module(object):
    
    # input : input to be passed into the layer
    def forward(self, *input): raise NotImplementedError
    
    # gradwrtoutput : gradient of the loss, with respect to the output of that layer (= input to next laxer)
    # return the gradient of the loss, with respect to the input of that layer (= output of previous layer)
    def backward(self, *gradwrtoutput): raise NotImplementedError
        
    def param(self): return []
    
    def to_string(self): raise NotImplementedError

In [4]:
# Hyperbolic tangent activation 
class Tanh(Module):
    
    def forward(self, *input):
        return input[0].tanh()
    
    # gradwrtoutput : predicted, target
    def backward(self, *gradwrtoutput):
        return 4 * (gradwrtoutput[0].exp() + gradwrtoutput[0].mul(-1).exp()).pow(-2)
    
    def to_string(self): return "TanH"
    

In [5]:
# ReLU activation
class Relu(Module):

    def forward(self, *input):
        x = input[0].clone()
        x[x < 0] = 0
        return x
    
    # gradwrtoutput : predicted, target
    def backward(self, *gradwrtoutput):
        x = gradwrtoutput[0]
        x[x < 0] = 0
        x[x > 0] = 1
        return x
    
    def to_string(self): return "ReLU"

In [46]:
# MSE Loss
class MSELoss(Module):
    
    # input: predicted, target
    def forward(self, *input):
        return (input[0] - input[1]).pow(2).sum()
    
    # gradwrtoutput : predicted, target
    def backward(self, *gradwrtoutput):
        return 2 * (gradwrtoutput[0] - gradwrtoutput[1])
    
    def to_string(self): return "MSE Loss"

In [33]:
# Linear module
class Linear(Module):
    def __init__(self, in_size, out_size):
        
        self.in_size = in_size
        self.out_size = out_size
        
        # initializing the weights
        stdv = 1. / math.sqrt(self.in_size)
        self.w = torch.empty(out_size, in_size).uniform_(-stdv, stdv)
        self.b = torch.empty(out_size).uniform_(-stdv, stdv)
        
        # initializing the gradients
        self.dw = torch.empty(out_size, in_size).zero_()
        self.db = torch.empty(out_size).zero_()
        
        self.in_value = torch.empty(in_size)
    
    # only with one input sample
    def forward(self, *input):
        self.in_value = input[0]
        return self.w.mv(input[0]) + self.b
    
    def backward(self, *gradwrtoutput):
        
        # gradient for that one sample
        dw_c = gradwrtoutput[0].view(-1, 1).mm(self.in_value.view(1, -1))
        db_c = gradwrtoutput[0]
        
        # updating stored gradients
        self.dw.add_(dw_c)
        self.db.add_(db_c)
        
        # gradient with respect to the input
        d_input = gradwrtoutput[0].view(1, -1).mm(self.w)
        
        return d_input[0]
    
    def param(self):
        return self.w, self.b, self.dw, self.db
    
    # updating the weights wrt to the step size and gradients stored, then set the gradients back to zero
    def update_param(self, step_size):
        self.w = self.w - step_size * self.dw
        self.b = self.b - step_size * self.db
        self.dw.zero_()
        self.db.zero_()
        
    def to_string(self):
        return "Linear ({}, {})".format(self.in_size, self.out_size)

In [36]:
# A sequence of modules
class Sequential(Module):
    def __init__(self, *Module):
        self.layers = Module[:-1]
        self.loss = Module[-1]
        
    def forward(self, *input):
        x = input[0]
        for m in self.layers: # not forwarding through the loss
            x = m.forward(x)
        return x
    
    def train(self, inputs, targets, epochs, step_size):
                
        for e in range(epochs):
            
            loss = 0
            # iterate over each sample and update weights at each iteration
            # todo : iterate over batches and update weights after each batch
            for input, target in zip(inputs.split(1), targets.split(1)):
                
                # computing predicted values and loss
                predicted = self.forward(input[0])
                loss = loss + self.loss.forward(target[0], predicted)
                
                # computing derivative of the loss between predicted and target
                x = self.loss.backward(predicted, target[0])
                
                # backpropagate the loss through the net, adding the gradients in linear modules
                for m in reversed(self.layers):
                    x = m.backward(x)
            
            # update weights in linear modules, set their gradients back to zero
            for m in self.layers:
                if isinstance(m, Linear):
                    m.update_param(step_size)
                        
            print("Epoch {} | Loss {}".format(e, loss))
    
    def print_param(self):
        for m in self.layers:
            if isinstance(m, Linear):
                params = m.param()
                print("*** ", m.to_string(), " ***")
                print(params[0])
                print(params[1])
                print()
            
    def describe(self):
        for m in self.layers:
            print(m.to_string())
        print(self.loss)
            
    # predictions using current weights in the modules
    # input : tensor, samples aligned along first dimension
    def predict(self, input):
        nb_samples = input.shape[0]
        pred = torch.empty(nb_samples)
        for i in range(nb_samples):
            pred[i] = self.forward(input[i])
        return pred

In [28]:
# generate 2d points in [0,1] squared, targets are 0 if point inside the circle of squared radius 1/2pi and 1 outside  
def generate_points(nb):
    input = torch.empty((nb, 2))
    output = torch.empty((nb, 1))
    for i in range(nb):
        x = random.uniform(0,1) - 0.5
        y = random.uniform(0,1) - 0.5
        t = int(2 * math.pi * (pow(x, 2) + pow(y, 2)) < 1)
        input[i] = torch.tensor([x + 0.5, y + 0.5])
        output[i] = torch.tensor([t])
    return input, output

In [37]:
# accuracy for classes prediction
def accuracy(model, input, classes):
    
    nb_samples = input.shape[0]
    pred = model.predict(input)
    pred_classes = torch.empty(nb_samples)
    
    for i in range(nb_samples):
        if pred[i] < 0.5:
            pred_classes[i] = 0
        else:
            pred_classes[i] = 1
    nb_errors = (pred_classes - classes[:,0]).type(torch.BoolTensor).sum().item()
    return (nb_samples - nb_errors) / nb_samples

In [41]:
model = Sequential(
    Linear(2, 4), 
    Relu(), 
    Linear(4, 4), 
    Relu(), 
    Linear(4, 1), 
    MSELoss())

input, classes = generate_points(1000)
model.train(input, classes, epochs=20, step_size = 1e-5)

Epoch 0 | Loss 250.77943420410156
Epoch 1 | Loss 250.69854736328125
Epoch 2 | Loss 250.62692260742188
Epoch 3 | Loss 250.5628204345703
Epoch 4 | Loss 250.50576782226562
Epoch 5 | Loss 250.4564971923828
Epoch 6 | Loss 250.41514587402344
Epoch 7 | Loss 250.37857055664062
Epoch 8 | Loss 250.35028076171875
Epoch 9 | Loss 250.32679748535156
Epoch 10 | Loss 250.3082275390625
Epoch 11 | Loss 250.29293823242188
Epoch 12 | Loss 250.28208923339844
Epoch 13 | Loss 250.27581787109375
Epoch 14 | Loss 250.2721710205078
Epoch 15 | Loss 250.2732696533203
Epoch 16 | Loss 250.27561950683594
Epoch 17 | Loss 250.2845916748047
Epoch 18 | Loss 250.2931365966797
Epoch 19 | Loss 250.30393981933594


In [42]:
accuracy(model, input, classes)

0.505

In [43]:
model.print_param()

***  Linear (2, 4)  ***
tensor([[ 0.1787,  0.5101],
        [-0.1952, -0.3676],
        [-0.6068, -0.4256],
        [-0.5307, -0.3213]])
tensor([-0.4418, -0.5074, -0.5249, -0.1481])

***  Linear (4, 4)  ***
tensor([[ 0.2896, -0.1897,  0.2434,  0.4830],
        [ 0.0534,  0.2047, -0.2097, -0.1412],
        [-0.0407, -0.0878,  0.3726,  0.2596],
        [-0.1956,  0.4058, -0.2857, -0.0497]])
tensor([-0.5291,  0.1656, -0.3605, -0.5378])

***  Linear (4, 1)  ***
tensor([[0.4219, 0.2803, 0.2251, 0.0230]])
tensor([0.4376])

